In [27]:
# importons les bibliothèques 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

from imblearn.over_sampling import SMOTE




In [28]:
# Chargeons nos données
df = pd.read_csv("../data/raw/healthcare-dataset-stroke-data.csv")

print(df.head())

      id  gender   age  ...   bmi   smoking_status stroke
0   9046    Male  67.0  ...  36.6  formerly smoked      1
1  51676  Female  61.0  ...   NaN     never smoked      1
2  31112    Male  80.0  ...  32.5     never smoked      1
3  60182  Female  49.0  ...  34.4           smokes      1
4   1665  Female  79.0  ...  24.0     never smoked      1

[5 rows x 12 columns]


In [29]:
# nous allons faire une copie de notre dataset pour éviter de l'endommager
df1 = df.copy()

In [30]:
# nous allons supprimer la colonne "id" qui n'est pas un paramètre médical ici, mais d'identification des patients de manière anonyme
df1 = df1.drop("id", axis=1)

print(df1.head())

   gender   age  hypertension  ...   bmi   smoking_status stroke
0    Male  67.0             0  ...  36.6  formerly smoked      1
1  Female  61.0             0  ...   NaN     never smoked      1
2    Male  80.0             0  ...  32.5     never smoked      1
3  Female  49.0             0  ...  34.4           smokes      1
4  Female  79.0             1  ...  24.0     never smoked      1

[5 rows x 11 columns]


#### Création des variables X et y 

In [31]:
# Défissons les variables X et y
X=df1.drop(columns="stroke") # variables explicatives
y= df1["stroke"]                # variable cible


Les caractéristiques (X) du patient nous permettrons de prédire la présence ou non d'un AVC (y) à l'aide d'un algorithme (Modèle du Machine Learning).

In [32]:
# vérifions
print(f" X se représente ainsi: \n {X.head()}")

print()

print(f" y se présente ainsi: \n {y.head()}")

 X se représente ainsi: 
    gender   age  hypertension  ...  avg_glucose_level   bmi   smoking_status
0    Male  67.0             0  ...             228.69  36.6  formerly smoked
1  Female  61.0             0  ...             202.21   NaN     never smoked
2    Male  80.0             0  ...             105.92  32.5     never smoked
3  Female  49.0             0  ...             171.23  34.4           smokes
4  Female  79.0             1  ...             174.12  24.0     never smoked

[5 rows x 10 columns]

 y se présente ainsi: 
 0    1
1    1
2    1
3    1
4    1
Name: stroke, dtype: int64


In [33]:
print(f"X est composé de: {X.shape[0]} lignes et {X.shape[1]} colonnes")
print(f"y est composé de: {y.shape[0]} lignes")

X est composé de: 5110 lignes et 10 colonnes
y est composé de: 5110 lignes


y = 5110 est une série, un seul vecteur de 5 110 valeurs, ce qui est exactement ce qu'attendent les modèles de scikit-learn en tant que variable cible.



#### Séparation du jeu d'entrainement et du jeu de test (jeu de test à 20% et random_state = 42)
Nous allons ajouter stratify = y car notre dataset contient un jeu de donnée déséquilibré :
95 % de non AVC et 5 % d'AVC

Comme nous allons séparer les données au hasard, on pourrait obtenir un jeu de test avec très peu de patients ayant subi un AVC. Le modèle serait alors mal évalué.

d'où pour conserver la même proportion d'AVC dans les deux jeux, on ajoutera stratify=y.

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
                                        X, 
                                        y, 
                                        test_size = 0.2, 
                                        random_state = 42,
                                        stratify = y
                                        )


In [35]:
# vérifions
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train:  (4088, 10)
X_test:  (1022, 10)
y_train:  (4088,)
y_test:  (1022,)


In [44]:
# vérifions que la proportion des classes est bien concervée
print((y_train.value_counts(normalize=True)*100).round())

stroke
0    95.0
1     5.0
Name: proportion, dtype: float64


In [46]:
print((y_test.value_counts(normalize=True)*100).round())

stroke
0    95.0
1     5.0
Name: proportion, dtype: float64


#### Feature Engineering (Identifier les types de variables)

In [48]:
# variables numériques

num_features = ["age", 
                "avg_glucose_level", 
                "bmi"]

# variables catégorielles

cat_features = ["gender", "ever_married", 
                "work_type", "Residence_type", 
                "smoking_status" ]

# variables numériques binaies

binary_features = ["hypertension", 
                    "heart_disease"]
                    



#### Nous allons nous intéresser à nos valeurs manquantes de la variable `bmi`
Etant donné que notre feature "bmi" a une distribution asymétrique, nous allons imputer nos valeurs manquantes avec la médiane qui est plus robuste que la moyenne.

Pour éviter le data leakage, nous allons utiliser la médiane de notre jeu d'entrainemeent puis l'appliquer directement à notre jeu de test 

In [51]:
# médiane jeu d'entrainement
median_bmi = X_train["bmi"].median()

median_bmi

np.float64(28.0)

In [52]:
# imputation médiane au train et test
X_train["bmi"] = X_train["bmi"].fillna(median_bmi)

X_test["bmi"] = X_test["bmi"].fillna(median_bmi)


In [53]:
# vérifions
print(X_train.isna().sum())

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
dtype: int64


In [54]:
print(X_test.isna().sum())

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
dtype: int64


### Choisir et importer un algorithme de ML (Algorithme de Machine Learning)